# LIAR — Full Explainability Pipeline

Complete pipeline for the LIAR dataset: load all three models, find prediction disagreements, apply SHAP and LIME to each model, compare the two explanation methods, then feed DistilBERT's SHAP attributions into Google's Gemini API (free tier) to generate plain-English explanations.

1. Get a free Gemini API key: https://aistudio.google.com/apikey 
2. `pip install shap lime google-genai`


## 1. Load All Three Models and Test Data

In [1]:
import pandas as pd
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

test_df = pd.read_csv("liar_test.csv")
train_df = pd.read_csv("liar_train.csv")

MAX_LENGTH = 64
print("Test set size:", len(test_df))


Using device: cuda
Test set size: 790


### 1.1 TF-IDF Baseline

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english", min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])
baseline_pipeline.fit(train_df["text"], train_df["label_id"])

tfidf_vectorizer = baseline_pipeline.named_steps["tfidf"]
logreg_clf = baseline_pipeline.named_steps["clf"]

def tfidf_predict_proba(texts):
    vecs = tfidf_vectorizer.transform(texts)
    return logreg_clf.predict_proba(vecs)

test_df["baseline_pred"] = baseline_pipeline.predict(test_df["text"].astype(str))
print(f"TF-IDF test accuracy: {(test_df['baseline_pred'] == test_df['label_id']).mean():.4f}")


TF-IDF test accuracy: 0.6241


### 1.2 DistilBERT

In [3]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained("./distilbert_liar_final")
distilbert_model = DistilBertForSequenceClassification.from_pretrained("./distilbert_liar_final")
distilbert_model.to(device)
distilbert_model.eval()

def distilbert_predict_proba(texts, batch_size=8):
    all_probs = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = distilbert_tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = distilbert_model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        del inputs, logits
        torch.cuda.empty_cache()
    return np.vstack(all_probs)

distilbert_probs = distilbert_predict_proba(test_df["text"].astype(str).tolist())
test_df["distilbert_pred"] = distilbert_probs.argmax(axis=1)
test_df["distilbert_confidence"] = distilbert_probs.max(axis=1)

print(f"DistilBERT test accuracy: {(test_df['distilbert_pred'] == test_df['label_id']).mean():.4f}")


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT test accuracy: 0.6886


### 1.3 RoBERTa

In [4]:
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

roberta_tokenizer = RobertaTokenizerFast.from_pretrained("./roberta_liar_final")
roberta_model = RobertaForSequenceClassification.from_pretrained("./roberta_liar_final")
roberta_model.to(device)
roberta_model.eval()

def roberta_predict_proba(texts, batch_size=8):
    all_probs = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = roberta_tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = roberta_model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        del inputs, logits
        torch.cuda.empty_cache()
    return np.vstack(all_probs)

roberta_probs = roberta_predict_proba(test_df["text"].astype(str).tolist())
test_df["roberta_pred"] = roberta_probs.argmax(axis=1)

print(f"RoBERTa test accuracy: {(test_df['roberta_pred'] == test_df['label_id']).mean():.4f}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RoBERTa test accuracy: 0.6076


## 2. Prediction Disagreements Across All Three Models

Find cases where predictions actually differ, then use SHAP/LIME to understand what's driving those differences — especially false positives.


In [5]:
test_df["disagree_baseline_distilbert"] = test_df["baseline_pred"] != test_df["distilbert_pred"]
test_df["disagree_baseline_roberta"] = test_df["baseline_pred"] != test_df["roberta_pred"]
test_df["disagree_distilbert_roberta"] = test_df["distilbert_pred"] != test_df["roberta_pred"]
test_df["disagree_any"] = (
    test_df["disagree_baseline_distilbert"] |
    test_df["disagree_baseline_roberta"] |
    test_df["disagree_distilbert_roberta"]
)

print(f"Baseline vs DistilBERT: {test_df['disagree_baseline_distilbert'].sum()} ({test_df['disagree_baseline_distilbert'].mean():.1%})")
print(f"Baseline vs RoBERTa: {test_df['disagree_baseline_roberta'].sum()} ({test_df['disagree_baseline_roberta'].mean():.1%})")
print(f"DistilBERT vs RoBERTa: {test_df['disagree_distilbert_roberta'].sum()} ({test_df['disagree_distilbert_roberta'].mean():.1%})")
print(f"\nAny disagreement: {test_df['disagree_any'].sum()} ({test_df['disagree_any'].mean():.1%})")

disagreement_df = test_df[test_df["disagree_any"]].copy()
disagreement_df[["text", "label", "baseline_pred", "distilbert_pred", "roberta_pred"]].head(10)


Baseline vs DistilBERT: 209 (26.5%)
Baseline vs RoBERTa: 269 (34.1%)
DistilBERT vs RoBERTa: 270 (34.2%)

Any disagreement: 374 (47.3%)


,text,label,baseline_pred,distilbert_pred,roberta_pred
1,Wisconsin is on pace to double the number of l...,fake,0,1,0
2,Says John McCain has done nothing to help the ...,fake,0,0,1
3,When asked by a reporter whether hes at the ce...,fake,1,1,0
6,Donald Trump is against marriage equality. He ...,real,0,0,1
10,Unfortunately we have documented instances whe...,fake,1,1,0
13,There have not been any public safety issues i...,real,0,1,0
14,The number of illegal immigrants could be 3 mi...,fake,0,1,0
15,Marijuana is less toxic than alcohol.,real,0,0,1
17,"Now, there was a time when someone like Scalia...",real,1,0,0
20,Its been since 1888 that a Senate of a differe...,fake,1,0,0


In [6]:
baseline_fp = test_df[(test_df["label_id"] == 0) & (test_df["baseline_pred"] == 1)]
distilbert_fp = test_df[(test_df["label_id"] == 0) & (test_df["distilbert_pred"] == 1)]
roberta_fp = test_df[(test_df["label_id"] == 0) & (test_df["roberta_pred"] == 1)]

print(f"Baseline false positives: {len(baseline_fp)}")
print(f"DistilBERT false positives: {len(distilbert_fp)}")
print(f"RoBERTa false positives: {len(roberta_fp)}")


Baseline false positives: 102
DistilBERT false positives: 133
RoBERTa false positives: 135


## 3. SHAP Explanations


In [7]:
import shap

example = disagreement_df.iloc[0] if len(disagreement_df) > 0 else test_df.iloc[0]
print("Example text:", example["text"])
print("True label:", example["label"])
print(f"Predictions — Baseline: {'fake' if example['baseline_pred']==1 else 'real'}, "
      f"DistilBERT: {'fake' if example['distilbert_pred']==1 else 'real'}, "
      f"RoBERTa: {'fake' if example['roberta_pred']==1 else 'real'}")


Example text: Wisconsin is on pace to double the number of layoffs this year.
True label: fake
Predictions — Baseline: real, DistilBERT: fake, RoBERTa: real


### 3.1 SHAP — TF-IDF Baseline (LinearExplainer)

In [8]:
background_texts = test_df["text"].astype(str).sample(min(100, len(test_df)), random_state=42)
background_vectors = tfidf_vectorizer.transform(background_texts)
tfidf_explainer = shap.LinearExplainer(logreg_clf, background_vectors)

def explain_tfidf_shap(text, top_k=5):
    vec = tfidf_vectorizer.transform([text])
    shap_values = tfidf_explainer.shap_values(vec)
    feature_names = tfidf_vectorizer.get_feature_names_out()
    values = shap_values[0] if len(shap_values.shape) == 2 else shap_values
    nonzero_idx = vec.nonzero()[1]
    contributions = [(feature_names[i], values[i]) for i in nonzero_idx]
    contributions.sort(key=lambda x: abs(x[1]), reverse=True)
    return contributions[:top_k]

tfidf_shap_tokens = explain_tfidf_shap(example["text"])
print("Top SHAP contributions (TF-IDF):")
for word, val in tfidf_shap_tokens:
    print(f"  {word}: {val:.4f} ({'toward FAKE' if val > 0 else 'toward REAL'})")


Top SHAP contributions (TF-IDF):
  wisconsin: 0.4910 (toward FAKE)
  year: -0.3014 (toward REAL)
  layoffs: 0.1649 (toward FAKE)
  number: -0.1357 (toward REAL)
  double: -0.0715 (toward REAL)


### 3.2 SHAP — DistilBERT

In [9]:
distilbert_masker = shap.maskers.Text(distilbert_tokenizer)
distilbert_explainer = shap.Explainer(distilbert_predict_proba, distilbert_masker)

def explain_distilbert_shap(text, class_idx, top_k=5):
    shap_values = distilbert_explainer([text])
    tokens = shap_values.data[0]
    values = shap_values.values[0, :, class_idx]
    pairs = list(zip(tokens, values))
    pairs.sort(key=lambda x: abs(x[1]), reverse=True)
    return pairs[:top_k]

distilbert_shap_tokens = explain_distilbert_shap(example["text"], class_idx=int(example["distilbert_pred"]))
print("Top SHAP tokens (DistilBERT):")
for token, val in distilbert_shap_tokens:
    print(f"  '{token.strip()}': {val:.4f} ({'toward FAKE' if val > 0 else 'toward REAL'})")


Top SHAP tokens (DistilBERT):
  'Wisconsin': 0.0696 (toward FAKE)
  'double': -0.0315 (toward REAL)
  'pace': -0.0266 (toward REAL)
  'year': -0.0258 (toward REAL)
  'lay': 0.0250 (toward FAKE)


### 3.3 SHAP — RoBERTa

In [10]:
roberta_masker = shap.maskers.Text(roberta_tokenizer)
roberta_explainer = shap.Explainer(roberta_predict_proba, roberta_masker)

def explain_roberta_shap(text, class_idx, top_k=5):
    shap_values = roberta_explainer([text])
    tokens = shap_values.data[0]
    values = shap_values.values[0, :, class_idx]
    pairs = list(zip(tokens, values))
    pairs.sort(key=lambda x: abs(x[1]), reverse=True)
    return pairs[:top_k]

roberta_shap_tokens = explain_roberta_shap(example["text"], class_idx=int(example["roberta_pred"]))
print("Top SHAP tokens (RoBERTa):")
for token, val in roberta_shap_tokens:
    print(f"  '{token.strip()}': {val:.4f} ({'toward FAKE' if val > 0 else 'toward REAL'})")


Top SHAP tokens (RoBERTa):
  'W': -0.0118 (toward REAL)
  'h': 0.0057 (toward FAKE)
  't': 0.0051 (toward FAKE)
  'c': -0.0042 (toward REAL)
  's': -0.0040 (toward REAL)


## 4. LIME Explanations

In [11]:
from lime.lime_text import LimeTextExplainer

lime_explainer = LimeTextExplainer(class_names=["real", "fake"])


### 4.1 LIME — TF-IDF Baseline

In [12]:
lime_tfidf = lime_explainer.explain_instance(example["text"], tfidf_predict_proba, num_features=5)

print("LIME explanation (TF-IDF):")
for word, weight in lime_tfidf.as_list():
    print(f"  {word}: {weight:.4f} ({'toward FAKE' if weight > 0 else 'toward REAL'})")


LIME explanation (TF-IDF):
  Wisconsin: 0.1644 (toward FAKE)
  year: -0.1028 (toward REAL)
  number: -0.0483 (toward REAL)
  layoffs: 0.0469 (toward FAKE)
  double: -0.0273 (toward REAL)


### 4.2 LIME — DistilBERT

In [13]:
lime_distilbert = lime_explainer.explain_instance(example["text"], distilbert_predict_proba, num_features=5, num_samples=500)

print("LIME explanation (DistilBERT):")
for word, weight in lime_distilbert.as_list():
    print(f"  {word}: {weight:.4f} ({'toward FAKE' if weight > 0 else 'toward REAL'})")


LIME explanation (DistilBERT):
  Wisconsin: 0.2116 (toward FAKE)
  double: -0.1651 (toward REAL)
  year: -0.1191 (toward REAL)
  pace: -0.0983 (toward REAL)
  layoffs: 0.0750 (toward FAKE)


### 4.3 LIME — RoBERTa

In [14]:
lime_roberta = lime_explainer.explain_instance(example["text"], roberta_predict_proba, num_features=5, num_samples=500)

print("LIME explanation (RoBERTa):")
for word, weight in lime_roberta.as_list():
    print(f"  {word}: {weight:.4f} ({'toward FAKE' if weight > 0 else 'toward REAL'})")


LIME explanation (RoBERTa):
  Wisconsin: 0.0313 (toward FAKE)
  layoffs: -0.0226 (toward REAL)
  the: -0.0223 (toward REAL)
  to: -0.0149 (toward REAL)
  of: -0.0138 (toward REAL)


## 5. SHAP vs LIME — Timing and Word Overlap Comparison

Runs both methods across several disagreement cases for DistilBERT (the model feeding the LLM pipeline), comparing generation time and word overlap.


In [15]:
import time

comparison_rows = []
n_cases = min(5, len(disagreement_df)) if len(disagreement_df) > 0 else 5
source_df = disagreement_df if len(disagreement_df) > 0 else test_df

for idx in range(n_cases):
    row = source_df.iloc[idx]

    start = time.time()
    shap_tokens = explain_distilbert_shap(row["text"], class_idx=int(row["distilbert_pred"]), top_k=5)
    shap_time = time.time() - start
    shap_words = set(t.strip().lower() for t, v in shap_tokens)

    start = time.time()
    lime_exp = lime_explainer.explain_instance(row["text"], distilbert_predict_proba, num_features=5, num_samples=500)
    lime_time = time.time() - start
    lime_words = set(w.strip().lower() for w, v in lime_exp.as_list())

    overlap = shap_words & lime_words
    overlap_pct = len(overlap) / max(len(shap_words | lime_words), 1)

    comparison_rows.append({
        "text_preview": row["text"][:80],
        "true_label": row["label"],
        "baseline_pred": "fake" if row["baseline_pred"] == 1 else "real",
        "distilbert_pred": "fake" if row["distilbert_pred"] == 1 else "real",
        "roberta_pred": "fake" if row["roberta_pred"] == 1 else "real",
        "shap_words": list(shap_words),
        "shap_time_sec": shap_time,
        "lime_words": list(lime_words),
        "lime_time_sec": lime_time,
        "word_overlap_pct": overlap_pct,
    })

shap_lime_comparison = pd.DataFrame(comparison_rows)
shap_lime_comparison.to_csv("liar_shap_lime_comparison.csv", index=False)
shap_lime_comparison


,text_preview,true_label,baseline_pred,distilbert_pred,roberta_pred,shap_words,shap_time_sec,lime_words,lime_time_sec,word_overlap_pct
0,Wisconsin is on pace to double the number of l...,fake,real,fake,real,"[lay, double, pace, year, wisconsin]",0.351302,"[layoffs, double, pace, year, wisconsin]",0.503865,0.666667
1,Says John McCain has done nothing to help the ...,fake,real,real,fake,"[john, mccain, ., says, vet]",0.320087,"[john, mccain, done, nothing, says]",0.464776,0.428571
2,When asked by a reporter whether hes at the ce...,fake,fake,fake,real,"[scott, walker, scheme, .]",0.736221,"[walker, scott, asked, nodded, when]",0.698822,0.285714
3,Donald Trump is against marriage equality. He ...,real,real,real,fake,"[., he, against, trump, donald]",0.289994,"[marriage, wants, against, trump, go]",0.485640,0.250000
4,Unfortunately we have documented instances whe...,fake,fake,fake,real,"[building, ted, unfortunately, people, documen...",0.645981,"[building, defecated, people, statehouse, docu...",0.538684,0.428571


## 6. Gemini API Setup

Free tier: 1,500 requests/day, 15 requests/minute, no credit card required.


In [16]:
from google import genai
import os

API_KEY = os.environ.get("GEMINI_API_KEY", "AQ.Ab8RN6Kkv38FIWJHo9_STGeuijYlLntPXFYffX62JfmukPtlhw")
client = genai.Client(api_key=API_KEY)
GEMINI_MODEL = "gemini-3.6-flash"

test_response = client.models.generate_content(model=GEMINI_MODEL, contents="Reply with exactly: API connection successful.")
print(test_response.text)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


API connection successful.


## 7. LLM Explanation Pipeline

Feeds DistilBERT's SHAP tokens (with direction) into Gemini, grounding the explanation in those specific words.


In [17]:
def build_explanation_prompt(text, predicted_label, confidence, top_tokens):
    fake_words = [t.strip() for t, v in top_tokens if v > 0]
    real_words = [t.strip() for t, v in top_tokens if v < 0]

    prompt = f"""The model classified this statement as [{predicted_label.upper()}] with {confidence:.0%} confidence.

Statement: "{text}"

The most influential words pushing toward FAKE were: {fake_words if fake_words else "none"}.
The most influential words pushing toward REAL were: {real_words if real_words else "none"}.

Write a 2-sentence explanation for a non-technical reader, referencing the specific influential words above. Do not introduce reasoning that isn't grounded in these words."""
    return prompt

def generate_explanation(prompt, retries=3):
    for attempt in range(retries):
        try:
            response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
            return response.text.strip()
        except Exception as e:
            if attempt < retries - 1:
                print(f"Retry {attempt + 1} after error: {e}")
                time.sleep(5)
            else:
                raise


## 8. Generate Explanations for Disagreement Cases

Adds a delay between calls to stay within the free tier's 15 requests/minute limit.


In [18]:
def explain_prediction(row_idx, df):
    row = df.iloc[row_idx]
    predicted_label = "fake" if row["distilbert_pred"] == 1 else "real"
    top_tokens = explain_distilbert_shap(row["text"], class_idx=int(row["distilbert_pred"]), top_k=5)

    prompt = build_explanation_prompt(row["text"], predicted_label, row["distilbert_confidence"], top_tokens)
    explanation = generate_explanation(prompt)

    print("Statement:", row["text"])
    print("True label:", row["label"], "| Predicted:", predicted_label, f"({row['distilbert_confidence']:.0%} confidence)")
    print("\nTop SHAP tokens:", [(t.strip(), round(v, 3)) for t, v in top_tokens])
    print("\nGemini Explanation:", explanation)
    print("-" * 100)

    time.sleep(4.5)
    return {"text": row["text"], "true_label": row["label"], "predicted_label": predicted_label,
            "top_tokens": top_tokens, "explanation": explanation}

source_for_explanations = disagreement_df if len(disagreement_df) > 0 else test_df
results = [explain_prediction(i, source_for_explanations) for i in range(min(5, len(source_for_explanations)))]


Statement: Wisconsin is on pace to double the number of layoffs this year.
True label: fake | Predicted: fake (59% confidence)

Top SHAP tokens: [('Wisconsin', np.float64(0.07)), ('double', np.float64(-0.031)), ('pace', np.float64(-0.027)), ('year', np.float64(-0.026)), ('lay', np.float64(0.025))]

Gemini Explanation: The model leaned toward labeling the statement as fake because specific words like "Wisconsin" and "lay" pushed the prediction toward that result. However, terms such as "double," "pace," and "year" suggested the statement was real, resulting in a slightly higher overall push toward a fake classification at 59% confidence.
----------------------------------------------------------------------------------------------------


Statement: Says John McCain has done nothing to help the vets.
True label: fake | Predicted: real (69% confidence)

Top SHAP tokens: [('McCain', np.float64(0.11)), ('.', np.float64(0.043)), ('Says', np.float64(0.034)), ('John', np.float64(0.014)), ('vet', np.float64(0.013))]

Gemini Explanation: The model ultimately classified the statement as REAL even though the words "Says", "John", "McCain", "vet", and the period "." were all identified as pushing the prediction toward FAKE. Because no words in the statement were found pushing toward a REAL rating, the model made its decision despite these specific terms leaning toward FAKE.
----------------------------------------------------------------------------------------------------


Statement: When asked by a reporter whether hes at the center of a criminal scheme to violate campaign laws, Gov. Scott Walker nodded yes.
True label: fake | Predicted: fake (66% confidence)

Top SHAP tokens: [('scheme', np.float64(0.017)), ('Scott', np.float64(0.017)), ('.', np.float64(0.016)), ('.', np.float64(0.015)), ('Walker', np.float64(0.015))]

Gemini Explanation: The model classified the statement as FAKE because specific words like "scheme", "Scott", and "Walker", along with periods ("."), strongly pointed toward it being false. At the same time, the model identified no words in the text that pushed the prediction toward being REAL.
----------------------------------------------------------------------------------------------------


Statement: Donald Trump is against marriage equality. He wants to go back.
True label: real | Predicted: real (72% confidence)

Top SHAP tokens: [('Trump', np.float64(0.105)), ('He', np.float64(0.04)), ('against', np.float64(0.028)), ('.', np.float64(0.025)), ('Donald', np.float64(0.025))]

Gemini Explanation: Even though the model ultimately labeled the statement as REAL, words such as 'Donald', 'Trump', 'He', 'against', and the period ('.') were actually pushing the prediction toward FAKE. Meanwhile, the model did not find any specific words that actively pushed the classification toward REAL.
----------------------------------------------------------------------------------------------------


Statement: Unfortunately we have documented instances where people defecated in the (Statehouse) building.
True label: fake | Predicted: fake (79% confidence)

Top SHAP tokens: [('building', np.float64(0.028)), ('Unfortunately', np.float64(0.027)), ('people', np.float64(0.025)), ('documented', np.float64(0.023)), ('ted', np.float64(0.018))]

Gemini Explanation: The model classified this statement as fake because words like "building," "Unfortunately," "people," "documented," and "ted" pushed its decision toward a false rating. At the same time, there were no influential words in the statement that suggested to the model that it was real.
----------------------------------------------------------------------------------------------------


In [19]:
import json

with open("liar_explanations_gemini.json", "w") as f:
    json.dump([
        {
            "text": r["text"], "true_label": r["true_label"], "predicted_label": r["predicted_label"],
            "top_tokens": [(t.strip(), float(v)) for t, v in r["top_tokens"]],
            "explanation": r["explanation"]
        } for r in results
    ], f, indent=2)

print("Saved: liar_explanations_gemini.json")


Saved: liar_explanations_gemini.json
